# Очистка train.csv

Ноутбук строит чистый слой техническидля основного event log

train.csv -> STG -> Data Quality -> ODS/CLEAN -> Parquet -> повторные DQ-проверки

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

#RAW_DIR = Path("data/raw")
RAW_DIR = Path("RiiidAnswerCorrectnessPrediction")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)


con = duckdb.connect()

def sql_path(path: Path) -> str:
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

def checkpoint(name, ok, detail=""):
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f" — {detail}"))

print("DuckDB:", duckdb.__version__)
print("RAW_DIR:", RAW_DIR.resolve())
print("OUT_DIR:", OUT_DIR.resolve())

DuckDB: 1.5.5
RAW_DIR: /Users/user/Desktop/project/RiiidAnswerCorrectnessPrediction
OUT_DIR: /Users/user/Desktop/project/data/processed


## 1. Пути и параметры

train.csv очень большой, поэтому мы не загружаем его целиком в pandas, DuckDB читает CSV потоково и записывает результат сразу в Parquet

In [3]:
TRAIN_PATH = RAW_DIR / "train.csv"
QUESTIONS_PATH = RAW_DIR / "questions.csv"
LECTURES_PATH = RAW_DIR / "lectures.csv"

TRAIN_CLEAN_PATH = OUT_DIR / "train_clean.parquet"
TRAIN_REJECTED_PATH = OUT_DIR / "train_rejected.parquet"
TRAIN_DUPLICATES_PATH = OUT_DIR / "train_duplicates.parquet"

assert TRAIN_PATH.exists(), f"Не найден файл: {TRAIN_PATH.resolve()}"

TRAIN = sql_path(TRAIN_PATH)
TRAIN_CLEAN = sql_path(TRAIN_CLEAN_PATH)
TRAIN_REJECTED = sql_path(TRAIN_REJECTED_PATH)
TRAIN_DUPLICATES = sql_path(TRAIN_DUPLICATES_PATH)

RUN_REFERENTIAL_CHECK = True

## 2. STG - читаем источник без изменения бизнес-смысла

Для train.csv задаём типы явно: на ~100 млн строк это заметно быстрее и экономнее памяти, чем читать всё как строки и потом приводить типы

In [4]:
con.execute(f"""
CREATE OR REPLACE VIEW stg_train AS
SELECT *
FROM read_csv(
    '{TRAIN}',
    header = true,
    nullstr = '',
    columns = {{
        'row_id': 'BIGINT',
        'timestamp': 'BIGINT',
        'user_id': 'INTEGER',
        'content_id': 'INTEGER',
        'content_type_id': 'TINYINT',
        'task_container_id': 'INTEGER',
        'user_answer': 'SMALLINT',
        'answered_correctly': 'SMALLINT',
        'prior_question_elapsed_time': 'DOUBLE',
        'prior_question_had_explanation': 'BOOLEAN'
    }}
)
""")

display(con.sql("SELECT * FROM stg_train LIMIT 10").df())

,row_id,timestamp,user_id,content_id,content_type_id,task_container_id,user_answer,answered_correctly,prior_question_elapsed_time,prior_question_had_explanation
0,0,0,115,5692,0,1,3,1,NaN,<NA>
1,1,56943,115,5716,0,2,2,1,37000.0,False
2,2,118363,115,128,0,0,0,1,55000.0,False
3,3,131167,115,7860,0,3,0,1,19000.0,False
4,4,137965,115,7922,0,4,1,1,11000.0,False
5,5,157063,115,156,0,5,2,1,5000.0,False
6,6,176092,115,51,0,6,0,1,17000.0,False
7,7,194190,115,50,0,7,3,1,17000.0,False
8,8,212463,115,7896,0,8,2,1,16000.0,False
9,9,230983,115,7863,0,9,0,1,16000.0,False


## 3. Data Quality до очистки

Используем те же классы проверок, что в DWH-пайплайне:

- Completeness — обязательные идентификаторы не пустые;
- Uniqueness — row_id уникален;
- Referential integrity — content_id должен существовать в соответствующем справочнике;
- Range / domain checks — допустимые значения признаков;
- Freshness здесь в календарном смысле неприменима: timestamp в Riiid — относительное время пользователя в миллисекундах, а не дата загрузки.

Отдельно проверяем семантику question/lecture-событий.

In [5]:
dq = con.sql("""
SELECT
    COUNT(*) AS rows_total,
    COUNT(DISTINCT row_id) AS unique_row_ids,

    COUNT(*) FILTER (
        WHERE row_id IS NULL
           OR timestamp IS NULL
           OR user_id IS NULL
           OR content_id IS NULL
           OR content_type_id IS NULL
           OR task_container_id IS NULL
    ) AS missing_mandatory,

    COUNT(*) FILTER (WHERE timestamp < 0) AS negative_timestamp,
    COUNT(*) FILTER (WHERE user_id < 0) AS negative_user_id,
    COUNT(*) FILTER (WHERE content_id < 0) AS negative_content_id,
    COUNT(*) FILTER (WHERE task_container_id < 0) AS negative_task_container_id,
    COUNT(*) FILTER (WHERE content_type_id NOT IN (0, 1)) AS bad_content_type,

    COUNT(*) FILTER (
        WHERE content_type_id = 0
          AND (user_answer NOT BETWEEN 0 AND 3
               OR answered_correctly NOT IN (0, 1))
    ) AS bad_question_semantics,

    COUNT(*) FILTER (
        WHERE content_type_id = 1
          AND (user_answer <> -1 OR answered_correctly <> -1)
    ) AS bad_lecture_semantics,

    COUNT(*) FILTER (
        WHERE prior_question_elapsed_time IS NOT NULL
          AND prior_question_elapsed_time < 0
    ) AS negative_prior_elapsed,

    MIN(timestamp) AS min_timestamp,
    MAX(timestamp) AS max_timestamp
FROM stg_train
""").df()

display(dq)

row = dq.iloc[0]
dup_rows = int(row["rows_total"] - row["unique_row_ids"])

checkpoint("Completeness", int(row["missing_mandatory"]) == 0,
           f"{int(row['missing_mandatory']):,} строк")
checkpoint("Uniqueness row_id", dup_rows == 0,
           f"{dup_rows:,} лишних строк по row_id")
checkpoint("content_type_id ∈ {0,1}", int(row["bad_content_type"]) == 0,
           f"{int(row['bad_content_type']):,} строк")
checkpoint("Семантика question-событий", int(row["bad_question_semantics"]) == 0,
           f"{int(row['bad_question_semantics']):,} строк")
checkpoint("Семантика lecture-событий", int(row["bad_lecture_semantics"]) == 0,
           f"{int(row['bad_lecture_semantics']):,} строк")
checkpoint("Временные значения неотрицательны",
           int(row["negative_timestamp"]) == 0 and int(row["negative_prior_elapsed"]) == 0,
           f"timestamp<0: {int(row['negative_timestamp']):,}; prior_elapsed<0: {int(row['negative_prior_elapsed']):,}")

,rows_total,unique_row_ids,missing_mandatory,negative_timestamp,negative_user_id,negative_content_id,negative_task_container_id,bad_content_type,bad_question_semantics,bad_lecture_semantics,negative_prior_elapsed,min_timestamp,max_timestamp
0,101230332,101230332,0,0,0,0,0,0,0,0,0,0,87425772049


✅ Completeness
✅ Uniqueness row_id
✅ content_type_id ∈ {0,1}
✅ Семантика question-событий
✅ Семантика lecture-событий
✅ Временные значения неотрицательны


## 4. Referential integrity

Проверка не зависит от уже очищенных справочников: используются исходные questions.csv и lectures.csv.


In [6]:
if RUN_REFERENTIAL_CHECK and QUESTIONS_PATH.exists() and LECTURES_PATH.exists():
    Q = sql_path(QUESTIONS_PATH)
    L = sql_path(LECTURES_PATH)

    con.execute(f"""
    CREATE OR REPLACE VIEW ref_questions AS
    SELECT question_id
    FROM read_csv_auto('{Q}', header=true)
    """)

    con.execute(f"""
    CREATE OR REPLACE VIEW ref_lectures AS
    SELECT lecture_id
    FROM read_csv_auto('{L}', header=true)
    """)

    ref_dq = con.sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE t.content_type_id = 0 AND q.question_id IS NULL
        ) AS missing_question_refs,
        COUNT(*) FILTER (
            WHERE t.content_type_id = 1 AND l.lecture_id IS NULL
        ) AS missing_lecture_refs
    FROM stg_train AS t
    LEFT JOIN ref_questions AS q
        ON t.content_type_id = 0
       AND t.content_id = q.question_id
    LEFT JOIN ref_lectures AS l
        ON t.content_type_id = 1
       AND t.content_id = l.lecture_id
    """).df()

    display(ref_dq)

    checkpoint("Referential integrity: questions",
               int(ref_dq.loc[0, "missing_question_refs"]) == 0,
               f"{int(ref_dq.loc[0, 'missing_question_refs']):,} ссылок")
    checkpoint("Referential integrity: lectures",
               int(ref_dq.loc[0, "missing_lecture_refs"]) == 0,
               f"{int(ref_dq.loc[0, 'missing_lecture_refs']):,} ссылок")
else:
    print("ℹ️ Referential integrity пропущена: questions.csv/lectures.csv не найдены или RUN_REFERENTIAL_CHECK=False")

,missing_question_refs,missing_lecture_refs
0,0,0


✅ Referential integrity: questions
✅ Referential integrity: lectures


## 5. Сохраняем подозрительные строки

Здесь откладываются только строки, нарушающие технические/семантические правила

Примечание: NULL в prior_question_* не считается ошибкой

In [7]:
invalid_predicate = """
       row_id IS NULL
    OR timestamp IS NULL
    OR user_id IS NULL
    OR content_id IS NULL
    OR content_type_id IS NULL
    OR task_container_id IS NULL
    OR timestamp < 0
    OR user_id < 0
    OR content_id < 0
    OR task_container_id < 0
    OR content_type_id NOT IN (0, 1)
    OR (content_type_id = 0 AND (
           user_answer NOT BETWEEN 0 AND 3
        OR answered_correctly NOT IN (0, 1)
       ))
    OR (content_type_id = 1 AND (
           user_answer <> -1
        OR answered_correctly <> -1
       ))
    OR (prior_question_elapsed_time IS NOT NULL
        AND prior_question_elapsed_time < 0)
"""

con.execute(f"""
COPY (
    SELECT *
    FROM stg_train
    WHERE {invalid_predicate}
)
TO '{TRAIN_REJECTED}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

rejected_n = con.sql(
    f"SELECT COUNT(*) FROM read_parquet('{TRAIN_REJECTED}')"
).fetchone()[0]

print(f"Подозрительных строк: {rejected_n:,}")
print("Файл:", TRAIN_REJECTED_PATH.resolve())

Подозрительных строк: 0
Файл: /Users/user/Desktop/project/data/processed/train_rejected.parquet


## 6. ODS/CLEAN — фильтрация + дедупликация

В официальном датасете row_id должен быть уникален, поэтому:

- если дублей нет — не запускаем тяжёлое оконное ранжирование
- иначе — оставляем одну строку на row_id, а все дубли сохраняем отдельно для аудита

In [ ]:
dup_row_ids = con.sql("""
SELECT COUNT(*) - COUNT(DISTINCT row_id)
FROM stg_train
""").fetchone()[0]

if dup_row_ids == 0:
    clean_query = f"""
    SELECT *
    FROM stg_train
    WHERE NOT ({invalid_predicate})
    """
    print("✅ Дубликатов row_id нет — дополнительная дедупликация не нужна.")
else:
    con.execute(f"""
    COPY (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY row_id
                    ORDER BY timestamp DESC, user_id, content_id
                ) AS rn
            FROM stg_train
            WHERE NOT ({invalid_predicate})
        )
        WHERE rn > 1
    )
    TO '{TRAIN_DUPLICATES}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    clean_query = f"""
    SELECT * EXCLUDE (rn)
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY row_id
                ORDER BY timestamp DESC, user_id, content_id
            ) AS rn
        FROM stg_train
        WHERE NOT ({invalid_predicate})
    )
    WHERE rn = 1
    """
    print(f"⚠️ Найдено лишних строк по row_id: {dup_row_ids:,}. Они дедуплицированы.")

con.execute(f"""
COPY (
    {clean_query}
)
TO '{TRAIN_CLEAN}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Готово:", TRAIN_CLEAN_PATH.resolve())

✅ Дубликатов row_id нет — дополнительная дедупликация не нужна.
Готово: /Users/user/Desktop/project/data/processed/train_clean.parquet


## 7. Финальные проверки очищенного слоя

In [9]:
final_dq = con.sql(f"""
SELECT
    COUNT(*) AS rows_total,
    COUNT(DISTINCT row_id) AS unique_row_ids,
    COUNT(*) FILTER (WHERE content_type_id NOT IN (0, 1)) AS bad_content_type,
    COUNT(*) FILTER (
        WHERE content_type_id = 0
          AND (user_answer NOT BETWEEN 0 AND 3
               OR answered_correctly NOT IN (0, 1))
    ) AS bad_question_semantics,
    COUNT(*) FILTER (
        WHERE content_type_id = 1
          AND (user_answer <> -1 OR answered_correctly <> -1)
    ) AS bad_lecture_semantics,
    COUNT(*) FILTER (
        WHERE timestamp < 0
           OR (prior_question_elapsed_time IS NOT NULL
               AND prior_question_elapsed_time < 0)
    ) AS bad_time_values,
    COUNT(*) FILTER (WHERE content_type_id = 0) AS question_events,
    COUNT(*) FILTER (WHERE content_type_id = 1) AS lecture_events
FROM read_parquet('{TRAIN_CLEAN}')
""").df()

display(final_dq)

r = final_dq.iloc[0]
checkpoint("Финал: row_id уникален",
           int(r["rows_total"]) == int(r["unique_row_ids"]))
checkpoint("Финал: домены корректны",
           int(r["bad_content_type"]) == 0
           and int(r["bad_question_semantics"]) == 0
           and int(r["bad_lecture_semantics"]) == 0
           and int(r["bad_time_values"]) == 0)

print(f"question events: {int(r['question_events']):,}")
print(f"lecture events:  {int(r['lecture_events']):,}")

,rows_total,unique_row_ids,bad_content_type,bad_question_semantics,bad_lecture_semantics,bad_time_values,question_events,lecture_events
0,101230332,101230332,0,0,0,0,99271300,1959032


✅ Финал: row_id уникален
✅ Финал: домены корректны
question events: 99,271,300
lecture events:  1,959,032


## 8. Результат

Основной очищенный файл: data/processed/train_clean.parquet

Подозрительные строки data/processed/train_rejected.parquet

Если в источнике были дубли row_id, они дополнительно попадут в data/processed/train_duplicates.parquet
